In [1]:
from pathlib import Path
import sys
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from scripts.rq3_function_lib import show_median_price_heatmap_per_region, mann_whitney_test_border_prices, show_border_price_difference, perform_matched_panel_regression_autobahn_stations
from scripts.rq3_function_lib import plot_yearly_autobahn_premium_line, plot_autobahn_premium_barchart
from scripts.rq3_function_lib import perform_wilcoxon_variance_test_on_autobahn, plot_wilcoxon_results_loolipop

In [2]:
region_price_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/regions_avg_prices_per_year')
region_path = Path(r'/Users/sebastian/data-science-projekt/plz_leitregionen.csv')
border_stations_file = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_border_stations.csv')
median_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/station_daily_mean_and_median_by_month')
non_autobahn_border_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/lower_non_autobahn_border_stations.csv')
stations_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/stations.csv')
panel_result_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv')
residual_error_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_residual_panel_error.parquet')
wilcoxon_result_path = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv')
monthly_median_prices_file = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_regional_median_prices_for_map.parquet')
median_border_distributions_file = Path(r'/Users/sebastian/data-science-projekt/rq_results/rq3_median_border_distributions.parquet')

### How much more expensive are Autobahn gas stations really?

The next part of the questions looks for the price differences between autobahn stations and regular stations. 

For that we use a matched panel regression that examines whether Autobahn stations have systematically higher fuel prices than nearby non-Autobahn stations. 

The model absorbs fixed regional and time effects. Additionaly, to mitigate brand effects, we categorize each station into brand, non-brand/free and unknown.

The models dependent variable is the daily station-level fuel price for a selected
fuel type and summary statistic. Letting $y_{i,t}$ denote the observed fuel
price of station $i$ on date $t$, the outcome in this analysis is one of

$$
y_{i,t} \in
\left\{
\text{diesel}_{i,t},
\text{e5}_{i,t},
\text{e10}_{i,t}
\right\}
$$

measured either as the daily **mean** or daily **median** price.

To make Autobahn and non-Autobahn stations geographically comparable, each
Autobahn station is matched with up to **5 nearest non-Autobahn stations**
within a maximum distance of **50 km**. This creates local comparison groups that reduce the bias from regional price differences.

The regression model from AbsorbingLS can be written in the general form

$$
y_i = x_i \beta + z_i \gamma + \epsilon_i
$$

where $y_i$ is the fuel price outcome, $x_i$ the regressor of
interest, and $z_i$ collects the fixed effects.

In our application, this corresponds to

$$
y_{i,t}
=
\beta \cdot \text{autobahn}_i
+ \theta^\top \text{brand\_category}_i
+ \alpha_m
+ \delta_t
+ \varepsilon_{i,t}
$$

where $\text{autobahn}_i$ is an indicator equal to 1 if station $i$ is
located on the Autobahn, $\text{brand\_category}_i$ captures the station's
brand classification, $\alpha_m$ denotes match-set (regional) fixed effects, and
$\delta_t$ denotes date fixed effects.

$\beta$ is the coefficient of interest. It captures the average price
difference between Autobahn and non-Autobahn stations after controlling for
local geographic proximity through the matched sets, for common daily shocks
through date fixed effects, and for systematic brand differences through brand
controls.

The model is estimated separately for each year and for each combination of
fuel type (**diesel**, **e5**, **e10**) and price statistic (**mean**,
**median**). Standard errors are clustered at the **station level** to account
for serial dependence in repeated observations of the same station over time.

We iterate over each fuel type, first performing the test on the mean prices and then to check the results, again over the median prices.

In [ ]:
""" autobahn_path = Path(r'/Users/sebastian/data-science-projekt/tankerkoenig_data/stations/autobahn_stations.csv')

fuel_types = ["diesel", "e5", "e10"]
statistics = ["mean", "median"]

test_summaries = []
analysis_panel_list = []
residuals_list = []

for fuel in fuel_types:
    for stat in statistics:
        
        res, panel, residuals = perform_matched_panel_regression_autobahn_stations(median_path,stations_path,autobahn_path,fuel,stat,return_residuals=True)
        test_summaries.append(res)
        analysis_panel_list.append(panel)
        residuals_list.append(residuals)



#residual_df = pd.concat(residuals_list, ignore_index=True)
#summary_df = pd.concat(test_summaries, ignore_index = True)
#print("\nSummary:")
#summary_df
#residual_df.to_parquet(r'/Users/sebastian/data-science-projekt/rq_results/rq3_residual_panel_error.parquet')
#summary_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_panel_summary.csv') """


In [13]:
summary_df = pd.read_csv(panel_result_path)
summary_df.head(15)

,Unnamed: 0,year,fuel_type,statistic,autobahn_coef,standard_error,p_value,ci_low,ci_high,n_observed
0,0,2014,diesel,mean,0.043297,0.001647,0.0,0.040070,0.046525,377130
1,1,2015,diesel,mean,0.051106,0.001983,0.0,0.047219,0.054994,685629
2,2,2016,diesel,mean,0.068193,0.002751,0.0,0.062802,0.073584,707278
3,3,2017,diesel,mean,0.080873,0.003013,0.0,0.074967,0.086778,716043
4,4,2018,diesel,mean,0.121318,0.003777,0.0,0.113914,0.128721,720505
5,5,2019,diesel,mean,0.157382,0.005040,0.0,0.147503,0.167261,711279
6,6,2020,diesel,mean,0.181175,0.006328,0.0,0.168773,0.193578,679417
7,7,2021,diesel,mean,0.193057,0.006648,0.0,0.180027,0.206087,667035
8,8,2022,diesel,mean,0.184888,0.006655,0.0,0.171844,0.197931,661864
9,9,2023,diesel,mean,0.240935,0.008872,0.0,0.223546,0.258324,652809


In [14]:
plot_yearly_autobahn_premium_line(summary_df)


In [15]:
plot_autobahn_premium_barchart(summary_df)

#### Results:

The results show a **positive and statistically significant Autobahn premium for fuel prices in every year** from 2014 to 2026. The estimated coefficient rises from about **€0.04 per liter in 2014** to about **€0.29 per liter in 2026**, which suggests that fuel prices at Autobahn stations were consistently higher than at matched non-Autobahn stations and that this price gap increased substantially over time.

The increase is especially visible from **2018 onward**, with another strong rise after **2022**. Since the confidence intervals remain clearly above zero in all years, the estimated premium appears to be very robust across specifications and years.

The regression suggests that even after controlling for **local matching structure**, **date fixed effects**, and **brand-category differences**, Autobahn stations tend to charge a noticeable price premium relative to comparable nearby stations.

Possible explanations for this may include:
- **weaker competitive pressure on the Autobahn**, since around 93% of the Autobahn service stations are owned by the (formerly government-owned and now private) Tank & Rast GmbH
- auction-based **supply and distribution rights** that oil companies have to obtain could contribute to higher fuel prices 
- the sharp increase in the later years may also be related to the broader energy market disruptions following the **war in Ukraine** and generally higher operating and service costs

At the same time, these regressions identify a **systematic association**, not a definitive causal mechanism. The proposed explanations are therefore plausible interpretations of the observed premium, but they are **not directly tested by this model**.





### Are the gas prices at Autobahn stations more volatile than the prices at normal gas stations?
Now we perform a wilcoxon signed rank test to see wether the autobahn station prices differ from normal station prices in volatility.

For this test, we examine whether the **residual price volatility** differs between
**Autobahn stations** and their matched **non-Autobahn controls** after the
fixed effects on the price have been removed in the panel
regression above.

The input for the test is the set of regression residuals
$\hat{\varepsilon}_{i,t}$ saved from the panel regession model. These residuals
represent the part of station-level fuel prices that remains unexplained after
controlling for the Autobahn factor, brand effects, regional fixed
effects, and date fixed effects.

For each station $i$ in year $y$, volatility is summarized from the residual
series using one of two volatility measures. If the robust option is selected,
volatility is measured by the **median absolute deviation**

$$
\text{median-absolute-deviation}_{i,y}
=
\operatorname{median}_{t \in y}
\left|
\hat{\varepsilon}_{i,t}
-
\operatorname{median}_{s \in y}(\hat{\varepsilon}_{i,s})
\right|
$$

Alternatively, volatility can be measured by the **standard deviation**

$$
\text{SD}_{i,y}
=
\sqrt{
\frac{1}{n_{i,y}-1}
\sum_{t \in y}
\left(
\hat{\varepsilon}_{i,t}
-
\bar{\hat{\varepsilon}}_{i,y}
\right)^2
}
$$

where $n_{i,y}$ is the number of residual observations for station $i$ in year
$y$. We will only consider stations with at least **30 observations** for the
volatility comparison.

To obtain a matched comparison, the volatility of the Autobahn station in match
set $m$ is compared with the average volatility of its matched non-Autobahn
control stations. Letting $V^T_{m,y}$ denote the Autobahn-station volatility and
$\bar{V}^C_{m,y}$ the mean control volatility in the same match set and year, we
define the paired difference as

$$
d_{m,y} = V^T_{m,y} - \bar{V}^C_{m,y}
$$

We implemented a two sided test for paired samples. In the present application, this is appropriate
because volatility is compared **within matched sets**, so the comparison is
based on paired Autobahn-control differences rather than on two independent
samples.

Following the formal definition, the two-sided test evaluates
whether the distribution of the paired differences $d_{m,y}$ is symmetric about
zero. The null and alternative hypotheses can be written as

$$
H_0: d_{m,y} \text{ is symmetrically distributed around } 0
$$

$$
H_1: d_{m,y} \text{ is not symmetrically distributed around } 0
$$

For the test, the nonzero absolute differences $|d_{m,y}|$ are ranked, their
original signs are reattached, and the Wilcoxon statistic is constructed from
the signed ranks. In the two-sided case, the test statistic is

$$
T = \min(W^+, W^-)
$$

where $W^+$ is the sum of ranks associated with positive differences and $W^-$
is the sum of ranks associated with negative differences. In our function,
pairs with $d_{m,y} = 0$ are removed before the test, so only non-zero matched
differences contribute to the statistic.

In our application, a **positive median difference** indicates that Autobahn
stations exhibit higher residual volatility than their matched non-Autobahn
controls in the same year. A **small p-value** suggests that the paired
volatility differences are not centered symmetrically around zero, which is
evidence of a systematic difference in residual volatility between the two
groups.


In [16]:
residual_df = pd.read_parquet(residual_error_path)

In [ ]:
""" wilcoxon_df = perform_wilcoxon_variance_test_on_autobahn(residual_df,measure="mad")

print(wilcoxon_df.head(15)) 
wilcoxon_df.to_csv(r'/Users/sebastian/data-science-projekt/rq_results/rq3_wilcoxon.csv')  """

    year measure  n_pairs  mean_test_volatility  mean_control_volatility  \
0   2014     mad      294              0.011654                 0.011069   
1   2015     mad      299              0.015624                 0.011839   
2   2016     mad      301              0.014081                 0.011854   
3   2017     mad      307              0.014609                 0.012092   
4   2018     mad      374              0.019972                 0.016912   
5   2019     mad      309              0.018834                 0.013823   
6   2020     mad      305              0.019792                 0.014994   
7   2021     mad      298              0.017602                 0.013051   
8   2022     mad      296              0.036291                 0.021432   
9   2023     mad      300              0.030703                 0.017223   
10  2024     mad      241              0.030864                 0.015777   
11  2025     mad      236              0.019933                 0.014305   
12  2026    

In [18]:
wilcoxon_df = pd.read_csv(wilcoxon_result_path)

wilcoxon_df

,Unnamed: 0,year,measure,n_pairs,mean_test_volatility,mean_control_volatility,median_difference,wilcoxon_stat,p_value
0,0,2014,mad,294,0.011654,0.011069,0.000352,18788.0,4.725826e-02
1,1,2015,mad,299,0.015624,0.011839,0.002557,7142.0,1.712751e-24
2,2,2016,mad,301,0.014081,0.011854,0.001489,12326.0,5.929404e-12
3,3,2017,mad,307,0.014609,0.012092,0.000848,15461.0,1.490211e-07
4,4,2018,mad,374,0.019972,0.016912,0.001742,23332.0,2.058752e-08
5,5,2019,mad,309,0.018834,0.013823,0.002479,11141.0,3.711624e-16
6,6,2020,mad,305,0.019792,0.014994,0.003261,7691.0,3.404717e-24
7,7,2021,mad,298,0.017602,0.013051,0.002046,8897.0,2.556332e-19
8,8,2022,mad,296,0.036291,0.021432,0.014303,1791.0,1.057824e-42
9,9,2023,mad,300,0.030703,0.017223,0.009268,2305.0,2.061131e-41


In [19]:
plot_wilcoxon_results_loolipop(wilcoxon_df)

#### Results:

The test results show that **Autobahn stations have statistically significant higher residual price volatility than matched non-Autobahn stations in every year** from 2014 to 2026. This is visible in the fact that the **median difference is positive in all years**, which means that the volatility measure based on the residuals is consistently larger for Autobahn stations than for their matched non-Autobahn control groups.

The effect is already present in the early years, but it becomes much stronger from **2022 to 2024**. In particular, the median volatility difference rises sharply in **2022** and remains clearly high in **2023** and **2024**. This suggests that Autobahn stations not only exhibit higher fuel prices on average, but also show more short-term price fluctuations that are not fully explained by the controls in the panel regression.

Because the test is based on the **residuals from the panel regression**, we can not not simply conclude that raw Autobahn prices fluctuate more, but rather that **the unexplained part of fuel price volatility** is larger at Autobahn stations even after controlling for match-set (regional) structure, date effects, and brand-category differences.

A plausible substantive interpretation (similar to the interpretation of the Autobahn premium) is that Autobahn stations may face a pricing environment with **weaker competitive pressure**, more rigid institutional structures (like the concession auctions), and potentially stronger exposure to market-wide shocks. The particularly large differences in **2022–2024** are consistent with the idea that periods of oil market disruption, such as the aftermath of the **war in Ukraine**, may have amplified this volatility gap. At the same time, these mechanisms are only possible explanations and are **not directly identified by the test itself**.

The later years should also be interpreted with some care, especially in 2026, since we only have the data for about 2 full months for far in this year.